# 03 — Reproduce the published NOWRITE result

**Stage:** Gautam's step 2, and the proposal's **Week 6 milestone**: *"Reproducing the
published 38–42% changed-answer rate under NOWRITE on the 3B checkpoints is the cleanest
available evidence that our instrumented fork behaves like the real system … we treat
failure as a blocker."*

Nothing in the 18 Aug pilot measures a task score. `AHN_algoverse.ipynb` cell 16 loops
over 20 RULER examples and reports `‖o_t‖` norms — a plumbing statistic, not an answer.
This notebook produces the first real numbers: mean F1, ΔF1, and **answer-change rate**
under AHN vs NOWRITE, on LongBench-E HotpotQA at the proposal's settings.

Target from the concurrent write-attrition study: **38–42% of answers change** while
mean F1 moves only **0.4–2.3 points**. Both halves matter — a large F1 swing would be as
much a red flag as no answer changes at all.

Cost: 60 examples × 2 conditions, ~4–32K tokens each. Budget 2–4 GPU-hours at 3B.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-4ca1/AHN
working directory pinned to /home/jupyter-dphs-4ca1/AHN


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
import torch, time, numpy as np
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok = bundle.tokenizer
print("loaded:", bundle.ahn_impl, "| window", bundle.sliding_window,
      "| sinks", bundle.num_attn_sinks)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded: GatedDeltaNet | window 8064 | sinks 128


In [4]:
from datasets import load_dataset

N_EXAMPLES = 60          # matches the concurrent study's cohort size
MAX_INPUT   = 32000
DATASET     = "hotpotqa"

ds = load_dataset("THUDM/LongBench", f"{DATASET}_e", split="test")
prompt_tmpl = (
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nThe following are given passages.\n{context}\n\n"
    "Answer the question based on the given passages. Only give me the answer and do "
    "not output any other words.\n\nQuestion: {input}\nAnswer:"
)

rows = []
for ex in ds:
    p = prompt_tmpl.format(context=ex["context"], input=ex["input"])
    n = len(tok.encode(p))
    if n > MAX_INPUT or n <= bundle.sliding_window + bundle.num_attn_sinks:
        continue          # AHN must actually be active, or the comparison is empty
    rows.append({"prompt": p, "answers": ex["answers"], "n_tokens": n,
                 "length_bucket": ex.get("length", None), "_id": ex.get("_id", len(rows))})

# length-stratified sample: shortest / median / longest thirds (proposal, RQ1 analysis)
rows.sort(key=lambda r: r["n_tokens"])
third = max(1, len(rows) // 3)
buckets = {"short": rows[:third], "mid": rows[third:2*third], "long": rows[2*third:]}
rng = np.random.default_rng(ai.SEED)
cohort = []
per = N_EXAMPLES // 3
for name, b in buckets.items():
    idx = rng.choice(len(b), size=min(per, len(b)), replace=False)
    for i in idx:
        r = dict(b[int(i)]); r["stratum"] = name; cohort.append(r)
print(f"eligible {len(rows)} -> cohort {len(cohort)}")
print({k: sum(1 for c in cohort if c["stratum"] == k) for k in buckets})
print("token range:", min(c["n_tokens"] for c in cohort), "-", max(c["n_tokens"] for c in cohort))


README.md: 0.00B [00:00, ?B/s]

LongBench.py: 0.00B [00:00, ?B/s]

The repository for THUDM/LongBench contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/THUDM/LongBench.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


data.zip:   0%|          | 0.00/114M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

eligible 182 -> cohort 60
{'short': 20, 'mid': 20, 'long': 20}
token range: 8491 - 17293


In [5]:
@torch.no_grad()
def predict(prompt, nowrite=False, max_new_tokens=32):
    ins = tok(prompt, return_tensors="pt").to(bundle.model.device)
    handles = []
    if nowrite:
        def z(m, i, o):
            return (torch.zeros_like(o[0]),) + o[1:] if isinstance(o, tuple) else torch.zeros_like(o)
        for L in bundle.ahn_layers:
            handles.append(bundle.model.model.layers[L].ahn.register_forward_hook(z))
    try:
        out = bundle.model.generate(**ins, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.eos_token_id)
    finally:
        for h in handles: h.remove()
    return tok.decode(out[0, ins["input_ids"].shape[1]:], skip_special_tokens=True).strip()

results, t0 = [], time.time()
for i, c in enumerate(cohort):
    a_on  = predict(c["prompt"], nowrite=False)
    a_off = predict(c["prompt"], nowrite=True)
    f1_on  = max(ai.qa_f1_score(a_on,  g) for g in c["answers"])
    f1_off = max(ai.qa_f1_score(a_off, g) for g in c["answers"])
    results.append({
        "id": c["_id"], "stratum": c["stratum"], "n_tokens": c["n_tokens"],
        "answer_ahn": a_on, "answer_nowrite": a_off,
        "f1_ahn": f1_on, "f1_nowrite": f1_off, "delta_f1": f1_on - f1_off,
        "answer_changed": ai.normalize_answer(a_on) != ai.normalize_answer(a_off),
        "gold": c["answers"],
    })
    if (i + 1) % 5 == 0:
        cc = np.mean([r["answer_changed"] for r in results])
        print(f"[{i+1:3d}/{len(cohort)}] changed={cc:.1%} "
              f"F1 {np.mean([r['f1_ahn'] for r in results]):.3f}/"
              f"{np.mean([r['f1_nowrite'] for r in results]):.3f} "
              f"({(time.time()-t0)/60:.1f} min)")
    ai.free_cuda()
print(f"\ndone in {(time.time()-t0)/60:.1f} min")


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

[  5/60] changed=100.0% F1 0.052/0.069 (1.1 min)


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

[ 10/60] changed=90.0% F1 0.143/0.168 (1.6 min)


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

...

[ 50/60] changed=92.0% F1 0.116/0.108 (7.2 min)


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

[ 55/60] changed=92.7% F1 0.122/0.140 (7.9 min)


/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/home/jupyter-dphs-4ca1/AHN/.ven

[ 60/60] changed=91.7% F1 0.115/0.130 (8.5 min)

done in 8.5 min


In [6]:
change_rate = float(np.mean([r["answer_changed"] for r in results]))
f1_on  = ai.bootstrap_ci([r["f1_ahn"] for r in results])
f1_off = ai.bootstrap_ci([r["f1_nowrite"] for r in results])
d_f1   = ai.bootstrap_ci([r["delta_f1"] for r in results])

repro = {
    "n": len(results),
    "answer_change_rate": change_rate,
    "answer_change_rate_ci": ai.bootstrap_ci([float(r["answer_changed"]) for r in results])[1:],
    "mean_f1_ahn": f1_on, "mean_f1_nowrite": f1_off, "delta_f1": d_f1,
    "delta_f1_points": d_f1[0] * 100,
    "published_change_rate_range": [0.38, 0.42],
    "published_f1_shift_points": [0.4, 2.3],
    "per_stratum": {
        s: {
            "n": sum(1 for r in results if r["stratum"] == s),
            "change_rate": float(np.mean([r["answer_changed"] for r in results if r["stratum"] == s])),
            "delta_f1": ai.bootstrap_ci([r["delta_f1"] for r in results if r["stratum"] == s]),
        } for s in ("short", "mid", "long")
    },
    "cfg": CFG, "dataset": DATASET, "max_input": MAX_INPUT,
}
repro["reproduction_ok"] = bool(0.30 <= change_rate <= 0.50
                                and abs(d_f1[0] * 100) <= 5.0)

ai.save_json({"summary": repro, "per_example": results}, "03_nowrite_reproduction.json")
print(json.dumps(repro, indent=2, default=str))
print("\nWEEK-6 MILESTONE:", "PASS" if repro["reproduction_ok"] else "FAIL — blocker, tell Gautam")


{
  "n": 60,
  "answer_change_rate": 0.9166666666666666,
  "answer_change_rate_ci": [
    0.85,
    0.9833333333333333
  ],
  "mean_f1_ahn": [
    0.11548439039585223,
    0.07877286178471035,
    0.16197583851039168
  ],
  "mean_f1_nowrite": [
    0.12986378290989722,
    0.07470622874356217,
    0.1958698909122009
  ],
  "delta_f1": [
    -0.014379392514045,
    -0.07323200675189166,
    0.03806633633763852
  ],
  "delta_f1_points": -1.4379392514045,
  "published_change_rate_range": [
    0.38,
    0.42
  ],
  "published_f1_shift_points": [
    0.4,
    2.3
  ],
  "per_stratum": {
    "short": {
      "n": 20,
      "change_rate": 0.95,
      "delta_f1": [
        0.025409177016728506,
        -0.0454778346621596,
        0.105525359607339
      ]
    },
    "mid": {
      "n": 20,
      "change_rate": 0.85,
      "delta_f1": [
        -0.006097402597402596,
        -0.11421731601731601,
        0.07147516233766231
      ]
    },
    "long": {
      "n": 20,
      "change_rate": 0.95

### Reading the result

- **Change rate 38–42%, |ΔF1| under ~2.5 points** → the instrumented fork behaves like
  the real system. This is the sentence that unblocks everything else, and it belongs in
  the next mentor update verbatim.
- **Change rate far below 38%** → AHN is barely active. Check `n_tokens` vs
  `sliding_window + num_attn_sinks`; the cohort filter above should have prevented this.
- **Change rate near 38% but ΔF1 large** → decoding config drift (sampling on, wrong
  chat template, wrong `max_new_tokens`). Compare against the upstream `eval/longbench`
  settings.

`per_example` in the saved JSON is the RQ3 join key: `delta_f1` per example is exactly
the outcome variable Table 8 correlates retention against. Keep it.
